# Thai Air Intelligence — Residual PM2.5 Dual-Model Trainer

Canonical Google Colab workflow for the same Python pipeline used by GitHub Actions and Production: 20 province-local **LightGBMRegressor** artifacts learn corrections to current-day persistence, while one pooled **RandomForestClassifier** predicts the five air-quality classes. Both tasks use a fixed 365-origin-date Validation window, a fixed 365-origin-date Test window, direct observed D+1 through D+7 targets, and a seven-day embargo on both boundaries. Regression correction weights are selected only on Validation, conservatively shrunk by 0.90, and frozen before Test.

Run every cell from top to bottom in a fresh Colab runtime. The notebook imports reviewed repository functions instead of maintaining a second copy of the training algorithm.

Required Colab Secrets: `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`. Forecast generation remains a separate Production operation.

Safe defaults: `REGISTER = False` and `ACTIVATE = False`. This creates auditable shadow artifacts locally but does not change `model_registry`, active models, or forecasts.

If regression training is ineligible or its serving artifact is unavailable, forecasts use `recent-mean-v1`: the arithmetic mean of the latest seven trusted observed PM2.5 days. Classification then derives its class from that numeric fallback.


In [ ]:
# 1. Configuration — keep both promotion switches False for shadow training
REGISTER = False
ACTIVATE = False
APPROVED_CODE_SHA = "2b5d895a7791600fd29c1fe3e4760dccd89b0be4"
PROVINCE = "all"  # "all" or one ID such as "TH-30" for diagnosis
MINIMUM_ROWS = 834  # 90 train + 365 validation + 365 test + 14 embargo
CV_SPLITS = 5
ARTIFACT_DIRECTORY = "training/artifacts"
ALLOWED_SOURCES = {"open-meteo"}

if ACTIVATE and not REGISTER:
    raise ValueError("ACTIVATE=True requires REGISTER=True")
if len(APPROVED_CODE_SHA) != 40:
    raise ValueError("APPROVED_CODE_SHA must be a reviewed 40-character commit SHA")
if MINIMUM_ROWS < 834:
    raise ValueError("Do not lower MINIMUM_ROWS below the reviewed 834-origin-date gate")


In [ ]:
# 2. Fetch the reviewed code and install only the missing training packages
# Move to /content first so rerunning this cell never deletes Colab's current working directory.
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_DIRECTORY = Path("/content/THAI-AIR-INTELLIGENCE-LITE")
os.chdir("/content")
shutil.rmtree(REPOSITORY_DIRECTORY, ignore_errors=True)

subprocess.run(
    ["git", "clone", "--filter=blob:none", "https://github.com/kzabCde/THAI-AIR-INTELLIGENCE-LITE.git", str(REPOSITORY_DIRECTORY)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPOSITORY_DIRECTORY), "checkout", "--detach", APPROVED_CODE_SHA],
    check=True,
)
checked_out_sha = subprocess.check_output(
    ["git", "-C", str(REPOSITORY_DIRECTORY), "rev-parse", "HEAD"],
    text=True,
).strip()
if checked_out_sha != APPROVED_CODE_SHA:
    raise RuntimeError(f"Expected {APPROVED_CODE_SHA}, checked out {checked_out_sha}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade-strategy",
        "only-if-needed",
        "supabase==2.31.0",
        "lightgbm==4.6.0",
        "joblib==1.4.2",
    ],
    check=True,
)

os.chdir(REPOSITORY_DIRECTORY)
repository_path = str(REPOSITORY_DIRECTORY)
sys.path = [path for path in sys.path if path != repository_path]
sys.path.insert(0, repository_path)
for module_name in list(sys.modules):
    if module_name in {"training", "api"} or module_name.startswith(("training.", "api.")):
        del sys.modules[module_name]
importlib.invalidate_caches()
print({"repository": repository_path, "approved_code_sha": checked_out_sha})

In [ ]:
# 3. Scientific-stack and native-model preflight
import platform
import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestClassifier

probe_X = np.asarray([[0.0], [1.0], [2.0], [3.0]], dtype=float)
LGBMRegressor(n_estimators=2, verbose=-1).fit(probe_X, probe_X[:, 0]).predict(probe_X[:1])
RandomForestClassifier(n_estimators=2, random_state=42).fit(probe_X, [1, 1, 2, 2]).predict_proba(probe_X[:1])
print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
})
print("Environment ABI check passed")

In [ ]:
# 4. Load the two server-side Supabase secrets without printing their values
import os
from google.colab import userdata

for secret_name in ("SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    secret_value = (userdata.get(secret_name) or "").strip()
    if not secret_value:
        raise ValueError(f"Missing Colab Secret: {secret_name}")
    os.environ[secret_name] = secret_value

if not os.environ["SUPABASE_URL"].startswith("https://"):
    raise ValueError("SUPABASE_URL must start with https://")
if os.environ["SUPABASE_SERVICE_ROLE_KEY"].lower().startswith(("sb_publishable_", "sb_anon_")):
    raise ValueError("Use a server-side service-role/secret key, not a public key")
print("Supabase secrets loaded")

In [ ]:
# 5. Import the exact residual-regression and pooled-classification pipeline
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

from training.dual_model_config import (
    FALLBACK_MODEL_NAME,
    FALLBACK_STRATEGY,
    FALLBACK_WINDOW_DAYS,
    POOLED_FEATURE_COLUMNS,
    POOLED_FEATURE_PROVENANCE,
    POOLED_FEATURE_VERSION,
    POOLED_PROVINCE_IDS,
    PipelineConfig,
)
from training.pm25_classes import CLASS_IDS, THRESHOLD_VERSION
from training.train_dual_models import (
    OBSERVED_VIEW,
    _json_safe,
    fetch_observed_rows,
    filter_training_rows,
)
from training.train_pooled_models import (
    CLASSIFICATION_MODEL_NAME,
    DIRECT_HORIZONS,
    REGRESSION_MODEL_NAME,
    PooledResult,
    build_pooled_examples,
    build_registry_rows,
    fetch_province_metadata,
    get_client,
    pooled_chronological_split,
    save_artifacts,
    train_classification,
    train_regression,
    upload_and_register,
)

config = PipelineConfig(
    minimum_rows=MINIMUM_ROWS,
    cv_splits=CV_SPLITS,
    artifact_directory=Path(ARTIFACT_DIRECTORY),
)
config.validate()
selected_provinces = tuple(POOLED_PROVINCE_IDS) if PROVINCE == "all" else (PROVINCE,)
if any(province_id not in POOLED_PROVINCE_IDS for province_id in selected_provinces):
    raise ValueError(f"Unknown province selection: {selected_provinces}")
print({
    "regression": REGRESSION_MODEL_NAME,
    "classification": CLASSIFICATION_MODEL_NAME,
    "fallback": {
        "model": FALLBACK_MODEL_NAME,
        "strategy": FALLBACK_STRATEGY,
        "window_days": FALLBACK_WINDOW_DAYS,
    },
    "features": len(POOLED_FEATURE_COLUMNS),
    "feature_version": POOLED_FEATURE_VERSION,
    "provinces": len(selected_provinces),
    "direct_horizons": list(DIRECT_HORIZONS),
    "register": REGISTER,
    "activate": ACTIVATE,
})


In [ ]:
# 6. Fetch trusted observations and inspect data quality by province
sb = get_client()
raw = fetch_observed_rows(sb, selected_provinces)
observed = filter_training_rows(raw, ALLOWED_SOURCES)
metadata = fetch_province_metadata(sb, selected_provinces)
quality = (
    observed.groupby("province_id", as_index=False)
    .agg(
        usable_days=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        minimum_trusted_hours=("trusted_hours", "min"),
        mean_pm25=("pm25_mean", "mean"),
    )
    .sort_values("province_id")
)
quality["has_minimum_origin_dates"] = quality["usable_days"].ge(MINIMUM_ROWS)
display(quality)
assert set(quality["province_id"]) == set(selected_provinces)
assert quality["minimum_trusted_hours"].ge(18).all()

In [ ]:
# 7. Build pooled D+1..D+7 examples from actual future PM2.5
examples = build_pooled_examples(observed, metadata)
example_counts = (
    examples.groupby(["province_id", "forecast_horizon_days"])
    .size()
    .unstack(fill_value=0)
)
class_distribution = (
    examples.groupby(["province_id", "target_air_quality_class"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_IDS, fill_value=0)
)
print("Pooled examples:", len(examples))
display(example_counts)
display(class_distribution)
assert examples.loc[:, POOLED_FEATURE_COLUMNS].notna().all(axis=None)
assert examples["target_pm25"].notna().all()

In [ ]:
# 8. Purged chronological split — every province on one date stays together
split = pooled_chronological_split(examples, config)
split_frames = {"train": split.train, "validation": split.validation, "test": split.test}
split_rows = []
date_sets = {}
for split_name, frame in split_frames.items():
    date_sets[split_name] = set(frame["date"].unique())
    split_rows.append({
        "split": split_name,
        "rows": len(frame),
        "origin_dates": frame["date"].nunique(),
        "first_origin": frame["date"].min(),
        "last_origin": frame["date"].max(),
        "last_target": frame["target_date"].max(),
        "provinces": frame["province_id"].nunique(),
    })
display(pd.DataFrame(split_rows))
print("Embargo dates removed:", len(split.dropped_embargo_dates), split.dropped_embargo_dates)
assert date_sets["train"].isdisjoint(date_sets["validation"])
assert date_sets["train"].isdisjoint(date_sets["test"])
assert date_sets["validation"].isdisjoint(date_sets["test"])
assert split.train["target_date"].max() < split.validation["date"].min()
assert split.validation["target_date"].max() < split.test["date"].min()

In [ ]:
# 9. Train province-local residual LightGBM models and preserve local eligibility evidence
regression = train_regression(split, config)
print(json.dumps(_json_safe({
    "model": REGRESSION_MODEL_NAME,
    "global_eligible": regression.global_eligible,
    "global_reasons": regression.global_reasons,
    "validation_metrics": regression.validation_metrics,
    "test_metrics": regression.test_metrics,
}), ensure_ascii=False, indent=2))
regression_by_province = pd.DataFrame([
    {"province_id": province_id, **metrics}
    for province_id, metrics in regression.province_metrics.items()
]).sort_values("province_id")
display(regression_by_province[[
    "province_id", "mae", "rmse", "r2", "skill_vs_persistence",
    "local_eligible", "global_gate_eligible", "eligibility_reasons",
]])


In [ ]:
# 10. Train pooled Random Forest classification and evaluate class imbalance
classification = train_classification(split, config)
metrics = classification.test_metrics
print(json.dumps(_json_safe({
    "model": CLASSIFICATION_MODEL_NAME,
    "global_eligible": classification.global_eligible,
    "global_reasons": classification.global_reasons,
    "accuracy": metrics.get("accuracy"),
    "balanced_accuracy": metrics.get("balanced_accuracy"),
    "macro_f1": metrics.get("macro_f1"),
    "brier_score": metrics.get("brier_score"),
    "expected_calibration_error": metrics.get("expected_calibration_error"),
}), ensure_ascii=False, indent=2))
per_class = pd.DataFrame([
    {"class_id": class_id, **metrics.get("per_class", {}).get(str(class_id), {})}
    for class_id in CLASS_IDS
])
display(per_class)
display(pd.DataFrame(
    metrics.get("confusion_matrix", []),
    index=[f"actual_{class_id}" for class_id in CLASS_IDS],
    columns=[f"predicted_{class_id}" for class_id in CLASS_IDS],
))

In [ ]:
# 11. Independent province/task eligibility and fallback plan
eligibility = pd.DataFrame([
    {
        "province_id": province_id,
        "regression_eligible": bool(regression.province_metrics[province_id]["eligible"]),
        "regression_skill": regression.province_metrics[province_id].get("skill_vs_persistence"),
        "classification_eligible": bool(classification.province_metrics[province_id]["eligible"]),
        "classification_macro_f1": classification.province_metrics[province_id].get("macro_f1"),
        "forecast_class_source": (
            "random_forest_classifier"
            if classification.province_metrics[province_id]["eligible"]
            else "lightgbm_regression_threshold"
            if regression.province_metrics[province_id]["eligible"]
            else "recent_mean_7d_fallback"
        ),
    }
    for province_id in selected_provinces
]).sort_values("province_id")
display(eligibility)

In [ ]:
# 12. Save exact native + portable artifacts and a reproducible run summary
run_id = str(uuid.uuid4())
audit = {
    "strategy": "pooled_split_local_residual_regression_and_pooled_classification",
    "pool_provinces": list(selected_provinces),
    "feature_version": POOLED_FEATURE_VERSION,
    "feature_provenance": POOLED_FEATURE_PROVENANCE,
    "target_source": "observed future PM2.5",
    "target_horizons": list(DIRECT_HORIZONS),
    "final_test_untouched_during_tuning": True,
    "same_date_same_partition": True,
    "embargo_days": max(DIRECT_HORIZONS),
    "dropped_embargo_dates": split.dropped_embargo_dates,
    "rows": {
        "train": len(split.train),
        "validation": len(split.validation),
        "test": len(split.test),
    },
}
registry_rows = build_registry_rows(
    run_id, selected_provinces, split, regression, classification, audit
)
result = PooledResult(
    run_id, selected_provinces, split, regression, classification, registry_rows, audit
)
artifacts = save_artifacts(result, config.artifact_directory, config)
run_summary = {
    "run_id": run_id,
    "mode": "register" if REGISTER else "shadow",
    "activate": ACTIVATE,
    "fallback": {
        "model": FALLBACK_MODEL_NAME,
        "strategy": FALLBACK_STRATEGY,
        "window_days": FALLBACK_WINDOW_DAYS,
        "used_for_provinces": eligibility.loc[
            ~eligibility["regression_eligible"], "province_id"
        ].tolist(),
    },
    "regression": {
        "model": REGRESSION_MODEL_NAME,
        "global_eligible": regression.global_eligible,
        "eligible_provinces": eligibility.loc[eligibility["regression_eligible"], "province_id"].tolist(),
        "metrics": regression.test_metrics,
    },
    "classification": {
        "model": CLASSIFICATION_MODEL_NAME,
        "global_eligible": classification.global_eligible,
        "eligible_provinces": eligibility.loc[eligibility["classification_eligible"], "province_id"].tolist(),
        "metrics": classification.test_metrics,
    },
    "audit": audit,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
summary_path = config.artifact_directory / run_id / "run_summary.json"
summary_path.write_text(
    json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("RUN_SUMMARY:", summary_path)
print(json.dumps(_json_safe(run_summary), ensure_ascii=False, indent=2))


In [ ]:
# 13. Optional promotion — uploads/registers candidates; activation is atomic across both tasks
if REGISTER:
    upload_and_register(sb, result, artifacts, activate=ACTIVATE)
    print({"registered": True, "activate_requested": ACTIVATE, "run_id": run_id})
else:
    print("REGISTER is False; no Storage, model_registry, activation, or forecast writes occurred.")


In [ ]:
# 14. Read-only verification: at most one active model per province and task
active_rows = (
    sb.table("model_registry")
    .select("province_id,task_type,model_name,model_family,run_id,eligibility_status,is_active")
    .eq("is_active", True)
    .order("province_id")
    .order("task_type")
    .execute()
    .data
    or []
)
active_models = pd.DataFrame(active_rows)
display(active_models)
if not active_models.empty:
    active_counts = active_models.groupby(["province_id", "task_type"]).size()
    assert int(active_counts.max()) <= 1

In [ ]:
# 15. Download the auditable shadow/registration artifact bundle
import shutil
from google.colab import files

archive_base = Path(f"pm25_pooled_dual_artifacts_{run_id}")
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=config.artifact_directory / run_id,
)
print("ZIP ready:", archive_path)
files.download(archive_path)